# Linear Regression: There is more than one way to skin a scatter plot.
In class, you learned the following procedure to fit a straigth line to data:

1. Plot the data. Make a scatter plot with the predictor (x) on the horizontal axis and the response (y) on the vertical axis. Look at it to judge whether the relationship is roughly linear.
2. Convert to standard units. For each variable, compute z = (x − mean) / SD, using the population SD from np.std.
3. Compute the correlation. r = mean(x_su × y_su).
4. Get the line in standard units. In standard units the regression line is y_su = r · x_su, so the slope is r.
5. Convert back to original units.
slope = r · SD_y / SD_x. Lab 8 explains this as stretching the axes.
intercept = mean_y − slope · mean_x. This works because the line always passes through (mean_x, mean_y).
6. Predict and check the fit.
Predictions: ŷ = slope · x + intercept.
7. Residuals: y − ŷ. Plot them and you should see no pattern and a flat trend.
8. Look at the coefficient of determination. r² is the fraction of the variation in y that the line explains.
8. Bootstrap the table, refit the line each time, and get confidence intervals for the slope and intercept.

In [ ]:
import numpy as np
from datascience import *
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('ggplot')

In [ ]:
faithful = Table.read_table("data/faithful-new.csv")
faithful.scatter('duration','wait')

In [ ]:
def standard_units(xyz):
    "Convert any array of numbers to standard units."
    return (xyz - np.mean(xyz))/np.std(xyz)  
def correlation(t, label_x, label_y):
    return np.mean(standard_units(t.column(label_x))*standard_units(t.column(label_y)))
    
# Regression
def slope(t, label_x, label_y):
    r = correlation(t, label_x, label_y)
    return r*np.std(t.column(label_y))/np.std(t.column(label_x))
def intercept(t, label_x, label_y):
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y)*np.mean(t.column(label_x))

In [ ]:
correlation(faithful,"duration","wait")

## Fitting a Line: Method 1 -- Converting to standard units
This is the method we taught you in class. 

In [ ]:
faithful.scatter('duration','wait')
slp = slope(faithful, 'duration','wait')
inter = intercept(faithful,'duration','wait')
print(f"Slope: {slp:.2f}    Intercept: {inter:.2f}")
plt.scatter(0,inter)
xs = [0, 7]
ys = [inter, slp * 7 + inter]
plt.plot(xs,ys, color='blue')
plt.savefig('faithful_regress.png')
plt.show()

## Fitting a Line: Method 2 -- Minimizing the residuals
This method uses the minimize function, which is part of the library for data science. First you create a function that calculates y, given a slope, intercept, and an x value. You apply that function to your dataset and minimize the residuals. 

In [ ]:
def mse_c(any_slope, any_intercept):
    tbl = faithful
    xlabel = 'duration'
    ylabel = 'wait'
    xdata, ydata = tbl.column(xlabel), tbl.column(ylabel)
    fitted = any_slope * xdata + any_intercept
    mse = np.mean((ydata - fitted) ** 2)
    print("Root mean squared error:", mse ** 0.5)
    return mse

In [ ]:
mse_c(5,5)

In [ ]:
mse_c(10.73,33.47)

In [ ]:
m_slope, m_intercept = minimize(mse_c)

In [ ]:
faithful.scatter('duration','wait')
plt.scatter(0,m_intercept)
xs = [0, 7]
ys = [inter, slp * 7 + inter]
plt.plot(xs,ys, color='green')
plt.savefig('faithful_regress_mse.png')
plt.show()

## Fitting a Line: Method 3 -- Using Numpy's Polyfit
The numpy module has a function to fit any polynomial to data. A polynomial of degree one is a straight line. 

In [ ]:
x = faithful.column("duration")
y = faithful.column("wait")
m, b = np.polyfit(x, y, 1)
print(f"The slope is: {m:.2f}, and the intercept is: {b:.2f}")

## Fitting a Line: Method 4 -- Using the Scipy module stats
The scipy module also has a function for fitting a straight line to data. 

In [ ]:
from scipy.stats import linregress

slope, intercept, r, p, stderr = linregress(x, y)
print(f"The slope is: {m:.2f}, and the intercept is: {b:.2f}")

## Fitting a Line: Method 5 -- Using the statsmodels module
The Stats-Models module has all sorts of statistical functions, including linear regression, using ordinary least squares. 

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(x)
model = sm.OLS(y, X).fit()
print(model.summary())

## Summary
Linear regression is such a common task there are many Python packages that can be used. The correct choice depends on your use case and preferences. **The key is that you understand what is happening under the hood when you fit a line to data.**